# Kaggle Regression: `property_organic_content`
---
**Target:** `property_organic_content` | **Metric:** RMSE | **ID:** `sample_id`  
**Strategy:** Weighted Ensemble (LightGBM + XGBoost + CatBoost) with StratifiedKFold

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings, os, gc, glob
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not installed - using LGB+XGB only")
SEED = 42; N_FOLDS = 5
TARGET = 'property_organic_content'; ID_COL = 'sample_id'
np.random.seed(SEED)
experiment_log = []
print("Libraries loaded")

## 1. Data Loading

In [ ]:
def load_data():
    # Kaggle (glob-safe)
    if os.path.isdir('/kaggle/input'):
        hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if hits:
            d = os.path.dirname(hits[0])
            return (pd.read_csv(os.path.join(d,'train.csv')),
                    pd.read_csv(os.path.join(d,'test.csv')),
                    pd.read_csv(os.path.join(d,'sample_submission.csv')))
    # Local
    for p in ['.','..','data']:
        if os.path.isfile(os.path.join(p,'train.csv')):
            return (pd.read_csv(os.path.join(p,'train.csv')),
                    pd.read_csv(os.path.join(p,'test.csv')),
                    pd.read_csv(os.path.join(p,'sample_submission.csv')))
    # Colab
    try:
        from google.colab import files
        print("Upload train.csv, test.csv, sample_submission.csv")
        uploaded = files.upload()
        from io import BytesIO
        return (pd.read_csv(BytesIO(uploaded['train.csv'])),
                pd.read_csv(BytesIO(uploaded['test.csv'])),
                pd.read_csv(BytesIO(uploaded['sample_submission.csv'])))
    except Exception:
        raise FileNotFoundError("CSV files not found.")
train_raw, test_raw, sample_sub = load_data()
print(f"Train: {train_raw.shape} | Test: {test_raw.shape} | Sub: {sample_sub.shape}")

## 2. Initial Data Audit

In [ ]:
print("=== Column Types ===")
print(train_raw.dtypes.value_counts())
print(f"\nTarget NaN: {train_raw[TARGET].isna().sum()}")
print(f"Target stats:\n{train_raw[TARGET].describe()}")
cat_cols_raw = train_raw.select_dtypes('object').columns.drop(ID_COL, errors='ignore').tolist()
num_cols_raw = train_raw.select_dtypes('number').columns.drop(TARGET, errors='ignore').tolist()
print(f"\nCategorical: {len(cat_cols_raw)} | Numeric: {len(num_cols_raw)}")
miss = pd.DataFrame({'train%': train_raw.isnull().mean(), 'test%': test_raw.isnull().mean()})
miss = miss.query('`train%`>0 or `test%`>0').sort_values('train%', ascending=False)
print(f"\nColumns with missing: {len(miss)}")
display(miss.style.format('{:.1%}').background_gradient(cmap='YlOrRd'))

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(train_raw[TARGET].dropna(), bins=60, edgecolor='k', alpha=.7, color='#4C72B0')
axes[0].set_title('Target Distribution')
axes[1].hist(np.log1p(train_raw[TARGET].dropna()), bins=60, edgecolor='k', alpha=.7, color='#55A868')
axes[1].set_title('log1p(Target)')
axes[2].boxplot(train_raw[TARGET].dropna(), vert=True); axes[2].set_title('Box Plot')
plt.tight_layout(); plt.show()
print(f"Skew: {train_raw[TARGET].skew():.3f} | Kurt: {train_raw[TARGET].kurtosis():.3f}")

In [ ]:
num_cols = train_raw.select_dtypes('number').columns.tolist()
corr = train_raw[num_cols].corrwith(train_raw[TARGET]).abs().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 5))
corr.plot.barh(ax=ax, color='#4C72B0'); ax.set_title('Top 20 |Corr| with Target')
plt.tight_layout(); plt.show()

In [ ]:
important_cats = ['biome','land_cover_type','parent_rock_type',
                  'geo_zone_macro','geo_zone_meso','geo_zone_micro',
                  'sampling_strategy','sampling_depth_cm','source_id']
important_cats = [c for c in important_cats if c in train_raw.columns]
n = len(important_cats)
rows = (n+2)//3
fig, axes = plt.subplots(rows, 3, figsize=(18, 4*rows))
for ax, c in zip(axes.flat, important_cats):
    grp = train_raw.groupby(c)[TARGET].median().sort_values()
    if len(grp) > 20: grp = grp.tail(20)
    grp.plot.barh(ax=ax, color='#DD8452'); ax.set_title(f'{c} (median target)')
for ax in axes.flat[n:]: ax.set_visible(False)
plt.tight_layout(); plt.show()

## 4. Feature Engineering

In [ ]:
def feature_engineering(df):
    df = df.copy()
    orig_cols = df.columns.tolist()
    # Original missing count (before any FE)
    important_miss = ['property_particle_coarse','property_particle_fine',
                      'property_acidity_index','cation_exchange_capacity',
                      'cation_Ca','cation_Mg','cation_Na','latitude','longitude']
    for c in important_miss:
        if c in df.columns:
            df[f'miss_{c}'] = df[c].isnull().astype(int)
    df['original_missing_count'] = df[orig_cols].isnull().sum(axis=1)

    band_a = [c for c in df.columns if 'band_A_PC' in c]
    band_b = [c for c in df.columns if 'band_B_PC' in c]
    for name, cols in [('A', band_a), ('B', band_b)]:
        if cols:
            df[f'band_{name}_mean'] = df[cols].mean(axis=1)
            df[f'band_{name}_std']  = df[cols].std(axis=1)
            df[f'band_{name}_min']  = df[cols].min(axis=1)
            df[f'band_{name}_max']  = df[cols].max(axis=1)
            df[f'band_{name}_range']= df[f'band_{name}_max'] - df[f'band_{name}_min']
            df[f'band_{name}_skew'] = df[cols].skew(axis=1)
            df[f'band_{name}_miss'] = df[cols].isnull().sum(axis=1)
    pc, pf = 'property_particle_coarse', 'property_particle_fine'
    if pc in df.columns and pf in df.columns:
        df['particle_ratio'] = df[pc] / (df[pf] + 1)
        df['particle_sum']   = df[pc] + df[pf]
    cation_cols = [c for c in df.columns if c.startswith('cation_') and c != 'cation_exchange_capacity']
    cec = 'cation_exchange_capacity'
    if cation_cols:
        df['cation_sum']  = df[cation_cols].sum(axis=1)
        df['cation_mean'] = df[cation_cols].mean(axis=1)
    if 'cation_Ca' in df.columns and 'cation_Mg' in df.columns:
        df['Ca_Mg_ratio'] = df['cation_Ca'] / (df['cation_Mg'] + 1e-6)
    if cec in df.columns:
        for cat in ['cation_Ca','cation_Mg','cation_Na']:
            if cat in df.columns:
                df[f'{cat.split("_")[1]}_CEC_ratio'] = df[cat] / (df[cec] + 1e-6)
    if pf in df.columns and cec in df.columns:
        df['fine_x_CEC'] = df[pf] * df[cec]
    if 'property_acidity_index' in df.columns and cec in df.columns:
        df['acidity_x_CEC'] = df['property_acidity_index'] * df[cec]
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df['lat_lon_interact'] = df['latitude'] * df['longitude']
        df['lat_abs'] = df['latitude'].abs()
        df['lon_abs'] = df['longitude'].abs()
    for band in ['A','B']:
        col = f'has_band_{band}_spectrum'
        if col in df.columns:
            df[f'has_band_{band}_flag'] = (df[col] == 'YES').astype(int)
    for c1, c2 in [('geo_zone_macro','biome'),('geo_zone_macro','parent_rock_type'),
                   ('biome','land_cover_type')]:
        if c1 in df.columns and c2 in df.columns:
            df[f'{c1}_x_{c2}'] = df[c1].astype(str) + '_' + df[c2].astype(str)
    return df

n_before = train_raw.shape[1]
train = feature_engineering(train_raw)
test  = feature_engineering(test_raw)
print(f"FE: {n_before} -> {train.shape[1]} features (+{train.shape[1]-n_before})")

## 5. Preprocessing & Validation Strategy

In [ ]:
drop_cols = [ID_COL, TARGET, 'has_band_A_spectrum', 'has_band_B_spectrum']
cat_base = ['source_id','sampling_strategy','sampling_depth_cm',
            'geo_zone_macro','geo_zone_micro','geo_zone_meso',
            'land_cover_type','biome','parent_rock_type']
geo_combos = [c for c in train.columns if '_x_' in c and train[c].dtype == 'object']
cat_all = [c for c in cat_base + geo_combos if c in train.columns]

# LGB/XGB: Label Encode
train_enc = train.copy(); test_enc = test.copy()
for c in cat_all:
    le = LabelEncoder()
    combined = pd.concat([train_enc[c], test_enc[c]]).astype(str).fillna('__NA__')
    le.fit(combined)
    train_enc[c] = le.transform(train_enc[c].astype(str).fillna('__NA__'))
    test_enc[c]  = le.transform(test_enc[c].astype(str).fillna('__NA__'))

# CatBoost: keep strings
train_cat = train.copy(); test_cat = test.copy()
for c in cat_all:
    train_cat[c] = train_cat[c].astype(str).fillna('__NA__')
    test_cat[c]  = test_cat[c].astype(str).fillna('__NA__')

# Frequency encoding for both
for c in cat_all:
    freq = pd.concat([train[c], test[c]]).astype(str).value_counts(normalize=True)
    for df in [train_enc, test_enc, train_cat, test_cat]:
        df[f'freq_{c}'] = train[c].astype(str).map(freq).fillna(0) if df is train_enc or df is train_cat else test[c].astype(str).map(freq).fillna(0)

feat_enc = [c for c in train_enc.columns if c not in drop_cols]
feat_cat = [c for c in train_cat.columns if c not in drop_cols]

X_enc = train_enc[feat_enc]; X_test_enc = test_enc[feat_enc]
X_cat = train_cat[feat_cat]; X_test_cat = test_cat[feat_cat]
y = train[TARGET].copy()
mask = y.notna()
X_enc, X_cat, y = X_enc[mask].reset_index(drop=True), X_cat[mask].reset_index(drop=True), y[mask].reset_index(drop=True)

y_binned = pd.qcut(y, q=N_FOLDS, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_enc, y_binned))
print(f"Feat LGB/XGB: {len(feat_enc)} | Feat CAT: {len(feat_cat)} | Samples: {len(y)}")
print(f"StratifiedKFold {N_FOLDS} folds (target-binned)")

## 6. Baseline Models

In [ ]:
# Mean baseline
oof_mean = np.full(len(y), y.mean())
rmse_mean = np.sqrt(mean_squared_error(y, oof_mean))
print(f"Mean Baseline RMSE: {rmse_mean:.5f}")
experiment_log.append({'model':'Mean Baseline','rmse':rmse_mean,'notes':'predict mean'})

# Ridge baseline
num_only = X_enc.select_dtypes('number').columns.tolist()
scaler = StandardScaler()
Xb = pd.DataFrame(scaler.fit_transform(X_enc[num_only].fillna(-999)), columns=num_only)
oof_ridge = np.zeros(len(y))
for ti, vi in folds:
    m = Ridge(alpha=10); m.fit(Xb.iloc[ti], y.iloc[ti])
    oof_ridge[vi] = m.predict(Xb.iloc[vi])
rmse_ridge = np.sqrt(mean_squared_error(y, oof_ridge))
print(f"Ridge Baseline RMSE: {rmse_ridge:.5f}")
experiment_log.append({'model':'Ridge Baseline','rmse':rmse_ridge,'notes':'numeric, scaled'})

## 7. Model Training

In [ ]:
lgb_params = dict(
    objective='regression', metric='rmse', n_estimators=2000,
    learning_rate=0.03, max_depth=7, num_leaves=63,
    subsample=0.7, colsample_bytree=0.6,
    reg_alpha=0.1, reg_lambda=1.0,
    min_child_samples=20, random_state=SEED, verbosity=-1, n_jobs=-1)
xgb_params = dict(
    objective='reg:squarederror', eval_metric='rmse', n_estimators=2000,
    learning_rate=0.03, max_depth=7, subsample=0.7, colsample_bytree=0.6,
    reg_alpha=0.1, reg_lambda=1.0, min_child_weight=20, random_state=SEED,
    tree_method='hist', n_jobs=-1, early_stopping_rounds=100)
cat_params = dict(
    iterations=2000, learning_rate=0.03, depth=7,
    l2_leaf_reg=3, random_seed=SEED, verbose=0, early_stopping_rounds=100)

oof_lgb = np.zeros(len(X_enc)); oof_xgb = np.zeros(len(X_enc))
oof_cat = np.zeros(len(X_cat)) if HAS_CATBOOST else None
pred_lgb = np.zeros(len(X_test_enc)); pred_xgb = np.zeros(len(X_test_enc))
pred_cat = np.zeros(len(X_test_cat)) if HAS_CATBOOST else None
fi_list = []

for fold, (ti, vi) in enumerate(folds):
    Xt, Xv, yt, yv = X_enc.iloc[ti], X_enc.iloc[vi], y.iloc[ti], y.iloc[vi]
    # LightGBM
    m1 = lgb.LGBMRegressor(**lgb_params)
    m1.fit(Xt, yt, eval_set=[(Xv, yv)],
           callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof_lgb[vi] = m1.predict(Xv); pred_lgb += m1.predict(X_test_enc)/N_FOLDS
    fi_list.append(pd.Series(m1.feature_importances_, index=feat_enc))
    # XGBoost
    m2 = xgb.XGBRegressor(**xgb_params)
    m2.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=0)
    oof_xgb[vi] = m2.predict(Xv); pred_xgb += m2.predict(X_test_enc)/N_FOLDS
    # CatBoost
    if HAS_CATBOOST:
        Xtc, Xvc = X_cat.iloc[ti], X_cat.iloc[vi]
        m3 = CatBoostRegressor(**cat_params)
        m3.fit(Xtc, yt, eval_set=(Xvc, yv), cat_features=cat_all, verbose=0)
        oof_cat[vi] = m3.predict(Xvc); pred_cat += m3.predict(X_test_cat)/N_FOLDS
    msg = f"Fold {fold+1} | LGB={np.sqrt(mean_squared_error(yv,oof_lgb[vi])):.5f}"
    msg += f" XGB={np.sqrt(mean_squared_error(yv,oof_xgb[vi])):.5f}"
    if HAS_CATBOOST: msg += f" CAT={np.sqrt(mean_squared_error(yv,oof_cat[vi])):.5f}"
    print(msg)

rmse_l = np.sqrt(mean_squared_error(y, oof_lgb))
rmse_x = np.sqrt(mean_squared_error(y, oof_xgb))
experiment_log.append({'model':'LightGBM','rmse':rmse_l,'notes':'2000 trees'})
experiment_log.append({'model':'XGBoost','rmse':rmse_x,'notes':'2000 trees'})
print(f"\nOOF: LGB={rmse_l:.5f} XGB={rmse_x:.5f}", end='')
if HAS_CATBOOST:
    rmse_c = np.sqrt(mean_squared_error(y, oof_cat))
    experiment_log.append({'model':'CatBoost','rmse':rmse_c,'notes':'native cats'})
    print(f" CAT={rmse_c:.5f}")
else:
    print()

## 8. Weighted Ensemble (OOF-optimized)

In [ ]:
def ens_rmse(ws, oofs, yt):
    return np.sqrt(mean_squared_error(yt, sum(w*o for w,o in zip(ws,oofs))))

oofs = [oof_lgb, oof_xgb] + ([oof_cat] if HAS_CATBOOST else [])
preds = [pred_lgb, pred_xgb] + ([pred_cat] if HAS_CATBOOST else [])
n_m = len(oofs)

best_r, best_w = 1e9, None
step = 0.05
if n_m == 3:
    for w1 in np.arange(0, 1.01, step):
        for w2 in np.arange(0, 1.01-w1, step):
            w3 = round(1-w1-w2, 2)
            r = ens_rmse([w1,w2,w3], oofs, y)
            if r < best_r: best_r, best_w = r, [w1,w2,w3]
else:
    for w1 in np.arange(0, 1.01, step):
        w2 = round(1-w1, 2)
        r = ens_rmse([w1,w2], oofs, y)
        if r < best_r: best_r, best_w = r, [w1,w2]

labels = ['LGB','XGB'] + (['CAT'] if HAS_CATBOOST else [])
print("Weights:", {l:round(w,3) for l,w in zip(labels, best_w)})
print(f"Ensemble OOF RMSE: {best_r:.5f}")
final_pred = sum(w*p for w,p in zip(best_w, preds))
oof_final  = sum(w*o for w,o in zip(best_w, oofs))
experiment_log.append({'model':'Weighted Ensemble','rmse':best_r,
                       'notes':str({l:round(w,3) for l,w in zip(labels,best_w)})})

## 9. Feature Importance

In [ ]:
fi = pd.concat(fi_list, axis=1).mean(axis=1).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 8))
fi.head(30).plot.barh(ax=ax, color='#4C72B0')
ax.set_title('Top 30 Feature Importances (LGB)'); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 10. Residual Analysis

In [ ]:
residuals = y - oof_final
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(oof_final, y, alpha=.15, s=5)
axes[0].plot([y.min(),y.max()],[y.min(),y.max()],'r--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual'); axes[0].set_title('Actual vs Predicted')
axes[1].hist(residuals, bins=60, edgecolor='k', alpha=.7, color='#DD8452')
axes[1].set_title('Residual Distribution')
axes[2].scatter(oof_final, residuals, alpha=.15, s=5); axes[2].axhline(0, color='r', ls='--')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Residual'); axes[2].set_title('Residuals vs Pred')
plt.tight_layout(); plt.show()

# Residual per target range
bins = pd.qcut(y, q=5, duplicates='drop')
res_by_bin = pd.DataFrame({'target_bin': bins, 'abs_residual': residuals.abs()})
print("\nRMSE per target range:")
for name, grp in res_by_bin.groupby('target_bin', observed=True):
    print(f"  {name}: RMSE={np.sqrt((grp['abs_residual']**2).mean()):.4f} n={len(grp)}")

## 11. Research Evidence & Experiment Log

In [ ]:
# Experiment log
print("=== Experiment Log ===")
exp_df = pd.DataFrame(experiment_log).sort_values('rmse')
display(exp_df)

In [ ]:
# FE impact: before vs after (compare Ridge baseline vs LGB)
print("=== Feature Engineering Impact ===")
print(f"Ridge (numeric only):    {rmse_ridge:.5f}")
print(f"LGB (with all FE):       {rmse_l:.5f}")
print(f"Improvement:             {rmse_ridge - rmse_l:.5f} ({(rmse_ridge-rmse_l)/rmse_ridge*100:.1f}%)")

In [ ]:
# Organic content per geo_zone_macro, meso, micro
print("=== Organic Content by Geographic Zones ===")
train_m = train_raw[train_raw[TARGET].notna()].copy()
for col in ['geo_zone_macro','geo_zone_meso','geo_zone_micro']:
    if col in train_m.columns:
        tbl = train_m.groupby(col)[TARGET].agg(['mean','median','std','count']).sort_values('mean', ascending=False)
        print(f"\n--- {col} ---")
        display(tbl)

In [ ]:
# Organic content per biome, land_cover_type
print("=== Organic Content by Biome & Land Cover ===")
for col in ['biome','land_cover_type']:
    if col in train_m.columns:
        tbl = train_m.groupby(col)[TARGET].agg(['mean','median','std','count']).sort_values('mean', ascending=False)
        print(f"\n--- {col} ---")
        display(tbl)
        fig, ax = plt.subplots(figsize=(10, 4))
        tbl['mean'].plot.barh(ax=ax, xerr=tbl['std'], color='#55A868')
        ax.set_title(f'{TARGET} by {col}'); plt.tight_layout(); plt.show()

In [ ]:
# Low acidity + low CEC analysis
print("=== Low Acidity + Low CEC Analysis ===")
if 'property_acidity_index' in train_m.columns and 'cation_exchange_capacity' in train_m.columns:
    ac = train_m['property_acidity_index']; cec = train_m['cation_exchange_capacity']
    low_ac = ac < ac.quantile(0.25); low_cec = cec < cec.quantile(0.25)
    groups = {'low_acid_low_cec': low_ac & low_cec, 'low_acid_high_cec': low_ac & ~low_cec,
              'high_acid_low_cec': ~low_ac & low_cec, 'high_acid_high_cec': ~low_ac & ~low_cec}
    rows = []
    for name, idx in groups.items():
        sub = train_m.loc[idx, TARGET]
        rows.append({'group':name,'mean':sub.mean(),'median':sub.median(),'std':sub.std(),'n':len(sub)})
    display(pd.DataFrame(rows))

In [ ]:
# Outlier combos: land_cover x geo_zone
print("=== Outlier Combinations: land_cover x geo_zone ===")
if 'land_cover_type' in train_m.columns and 'geo_zone_macro' in train_m.columns:
    combo = train_m.groupby(['land_cover_type','geo_zone_macro'])[TARGET].agg(['mean','std','count'])
    combo = combo[combo['count'] >= 5].sort_values('mean', ascending=False)
    print("Top 10 highest organic content combos:")
    display(combo.head(10))
    print("\nTop 10 lowest:")
    display(combo.tail(10))

In [ ]:
# High correlation pairs |corr| > 0.8
print("=== Highly Correlated Feature Pairs (|r| > 0.8) ===")
num_feats = X_enc.select_dtypes('number').columns
corr_matrix = X_enc[num_feats].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(upper.columns[j], upper.index[i], upper.iloc[i,j])
             for i in range(len(upper)) for j in range(len(upper.columns))
             if upper.iloc[i,j] > 0.8]
high_corr_df = pd.DataFrame(high_corr, columns=['feat1','feat2','corr']).sort_values('corr', ascending=False)
print(f"Pairs with |corr| > 0.8: {len(high_corr_df)}")
display(high_corr_df.head(20))

In [ ]:
# Train vs Test distribution comparison
print("=== Train vs Test Distribution ===")
common_num = [c for c in X_enc.select_dtypes('number').columns if c in X_test_enc.columns]
diffs = []
for c in common_num:
    tr_m, te_m = X_enc[c].mean(), X_test_enc[c].mean()
    tr_s, te_s = X_enc[c].std(), X_test_enc[c].std()
    mean_diff = abs(tr_m - te_m) / (tr_s + 1e-8)
    diffs.append({'feature':c, 'train_mean':tr_m, 'test_mean':te_m, 'norm_diff':mean_diff})
dist_df = pd.DataFrame(diffs).sort_values('norm_diff', ascending=False)
print("Top 15 features with largest train-test distribution shift:")
display(dist_df.head(15))

# Plot top 6
top6 = dist_df.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.flat, top6):
    ax.hist(X_enc[c].dropna(), bins=40, alpha=.5, label='train', density=True)
    ax.hist(X_test_enc[c].dropna(), bins=40, alpha=.5, label='test', density=True)
    ax.set_title(c); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Missing pattern correlation with residual
print("=== Missing Indicators vs Residuals ===")
miss_cols = [c for c in X_enc.columns if c.startswith('miss_') or c == 'original_missing_count']
if miss_cols:
    miss_corr = X_enc[miss_cols].corrwith(pd.Series(residuals.values, index=X_enc.index)).abs().sort_values(ascending=False)
    display(miss_corr.to_frame('|corr_with_residual|'))

# Per-category RMSE
print("\n=== Per-Category RMSE ===")
for c in ['biome','geo_zone_macro','parent_rock_type','sampling_depth_cm']:
    if c in train.columns:
        rows = []
        vals = train.loc[mask, c].reset_index(drop=True)
        for val in vals.unique():
            idx = vals == val
            if idx.sum() > 10:
                rows.append({'value':val, 'rmse':np.sqrt(mean_squared_error(y[idx], oof_final[idx])), 'n':idx.sum()})
        if rows:
            print(f"\n{c}:")
            display(pd.DataFrame(rows).sort_values('rmse', ascending=False).head(10))

## 12. Generate Submission

In [ ]:
submission = sample_sub[[ID_COL]].copy()
submission[TARGET] = final_pred
submission[TARGET] = submission[TARGET].clip(lower=0)

assert len(submission) == len(sample_sub), "Row count mismatch!"
assert submission[TARGET].isna().sum() == 0, "NaN found!"
assert (submission[TARGET] < 0).sum() == 0, "Negative preds!"
assert list(submission[ID_COL]) == list(sample_sub[ID_COL]), "ID order mismatch!"

submission.to_csv('submission.csv', index=False)
print(f"Saved submission.csv | Shape: {submission.shape}")
print(f"Min={submission[TARGET].min():.4f} Max={submission[TARGET].max():.4f} Mean={submission[TARGET].mean():.4f}")
display(submission.head(10))

## Summary

| Item | Detail |
|------|--------|
| CV | StratifiedKFold 5-fold (target-binned) |
| Models | LightGBM + XGBoost + CatBoost (native cats) |
| Ensemble | OOF-optimized weighted blend |
| FE | Spectral agg, cation ratios, missing indicators, freq encoding, geo combos |
| Post-proc | Clip >= 0 |
| Evidence | Geo/biome/land analysis, correlation pairs, train-test shift, residuals by range |

> All research evidence tables saved above. Research questions can be answered from these outputs.